# GMM ve OCSVM ile Credit Card Fraud Tespiti

Bu notebook, Credit Card veri seti üzerinde GMM ve One-Class SVM modellerini eğitir.

## Modeller:
- **One-Class SVM (OCSVM)**: RBF kernel ile anomali tespiti
- **Gaussian Mixture Model (GMM)**: Olasılıksal anomali tespiti

## Eğitim Yaklaşımları:
- **Majority (Normal)**: Sadece normal işlemlerle eğitim
- **Minority (Fraud)**: Sadece fraud işlemlerle eğitim

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import OneClassSVM
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

print("Kütüphaneler yüklendi.")

Kütüphaneler yüklendi.


## 1. Credit Card Verisini Yükle

In [2]:
# Credit Card verisini yükle
cc_df = pd.read_csv("../data/credit_card_PCA.csv")
print(f"Veri seti yüklendi: {cc_df.shape}")
print(cc_df.head())

Veri seti yüklendi: (284807, 30)
        PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
0  0.254214  0.176580 -0.179470 -0.161785  0.389370 -0.196259  0.745975   
1  0.404755 -0.021151 -0.155891 -0.415824 -0.401750  0.301347 -0.353866   
2 -0.369276  1.325235  0.832810  0.714899 -0.876420 -0.065279 -0.787775   
3 -0.262427  2.531671  1.319002 -0.603730 -0.329415  0.692313  0.086356   
4 -0.447496  0.111555 -0.359132  0.459141  0.683895 -0.176335  0.364476   

        PC8       PC9      PC10  ...      PC21      PC22      PC23      PC24  \
0 -0.539874  0.110176 -1.246900  ... -0.753015  0.646592  0.138289  0.457039   
1  0.001194  0.404043  0.398631  ... -0.286525  0.015835 -0.142906 -1.114539   
2 -1.968873 -0.468434 -1.787015  ...  1.933966  1.008190  0.687896 -0.413412   
3  0.438731  0.210016 -1.272721  ...  1.744840 -0.405084  0.484261 -0.426900   
4  0.087291  0.989826 -0.700293  ... -0.420311  0.132960 -0.847668  0.021658   

       PC25      PC26      PC27    

In [3]:
# Sütun isimlerini kontrol et
print("Sütunlar:")
print(cc_df.columns.tolist())

Sütunlar:
['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19', 'PC20', 'PC21', 'PC22', 'PC23', 'PC24', 'PC25', 'PC26', 'PC27', 'PC28', 'Amount', 'Class']


## 2. Veri Bölme

In [4]:
# Feature matrix ve target vektörü ayır
X_cc = cc_df.drop(columns=['Class'])
y_cc = cc_df['Class']

print("=== Credit Card Veri Seti ===")
print(f"X_cc boyutu: {X_cc.shape}")
print(f"Sınıf dağılımı:\n{y_cc.value_counts(normalize=True)}")

=== Credit Card Veri Seti ===
X_cc boyutu: (284807, 29)
Sınıf dağılımı:
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


## 3. Çapraz Doğrulama Ayarları

In [5]:
# StratifiedKFold ile sınıf dengesini koru
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
repeats = 5  # Tekrar sayısı

print(f"K-Fold: 5, Tekrar: {repeats}")

K-Fold: 5, Tekrar: 5


In [6]:
# Sonuçları saklamak için dictionary
results_cc = {
    'majority': {
        'OCSVM': [],
        'OCSVM_Sigmoid': [],
        'OCSVM_Isotonic': [],
        'GMM': [],
        'GMM_Sigmoid': []
    },
    'minority': {
        'OCSVM': [],
        'OCSVM_Sigmoid': [],
        'OCSVM_Isotonic': [],
        'GMM': [],
        'GMM_Sigmoid': []
    },
}
print("Sonuç dictionary'si oluşturuldu.")

Sonuç dictionary'si oluşturuldu.


## 4. Kalibrasyon Fonksiyonu

In [7]:
def calibrate_scores(scores, y_true, method="sigmoid"):
    """
    Anomali skorlarını kalibre eder.
    
    Args:
        scores: Ham anomali skorları
        y_true: Gerçek etiketler
        method: 'sigmoid' veya 'isotonic'
    
    Returns:
        Kalibre edilmiş model
    """
    if method == "sigmoid":
        calibrator = LogisticRegression(solver="lbfgs")
    elif method == "isotonic":
        calibrator = IsotonicRegression(out_of_bounds="clip")
    else:
        raise ValueError("Bilinmeyen kalibrasyon yöntemi.")

    calibrator.fit(scores.reshape(-1, 1), y_true)
    return calibrator

print("Kalibrasyon fonksiyonu tanımlandı.")

Kalibrasyon fonksiyonu tanımlandı.


## 5. Model Eğitimi

In [8]:
# CREDIT CARD: MAJORITY (Normal) vs MINORITY (Fraud) EĞİTİMİ

for r in range(repeats):
    print(f"\n--- Credit Card: Tekrar {r+1}/{repeats} ---")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + r)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_cc, y_cc), start=1):
        
        # Eğitim ve test setlerini ayır
        X_train, X_test = X_cc.values[train_idx], X_cc.values[test_idx]
        y_train, y_test = y_cc.values[train_idx], y_cc.values[test_idx]

        # ---------------------------------
        # 1) Majority eğitimi: sadece normal (y_train == 0)
        # ---------------------------------
        X_train_major = X_train[y_train == 0]

        # -- One-Class SVM (Majority) --
        ocsvm_major = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
        ocsvm_major.fit(X_train_major)

        # Anomali skorlarını al
        svm_scores_raw = ocsvm_major.decision_function(X_test)
        svm_anomaly = -svm_scores_raw

        # Kalibre et
        calibrator_sigmoid = calibrate_scores(svm_anomaly, y_test, method="sigmoid")
        calibrator_isotonic = calibrate_scores(svm_anomaly, y_test, method="isotonic")

        # Kalibre edilmiş olasılıkları al
        prob_sigmoid = calibrator_sigmoid.predict_proba(svm_anomaly.reshape(-1, 1))[:, 1] if hasattr(calibrator_sigmoid, "predict_proba") else calibrator_sigmoid.predict(svm_anomaly.reshape(-1, 1))
        prob_isotonic = calibrator_isotonic.predict(svm_anomaly.reshape(-1, 1))

        # Değerlendir
        auc_sigmoid = roc_auc_score(y_test, prob_sigmoid)
        aupr_sigmoid = average_precision_score(y_test, prob_sigmoid)
        auc_isotonic = roc_auc_score(y_test, prob_isotonic)
        aupr_isotonic = average_precision_score(y_test, prob_isotonic)

        results_cc['majority']['OCSVM_Sigmoid'].append((auc_sigmoid, aupr_sigmoid))
        results_cc['majority']['OCSVM_Isotonic'].append((auc_isotonic, aupr_isotonic))

        # -- GMM (Majority) --
        gmm_major = GaussianMixture(n_components=1, random_state=42)
        gmm_major.fit(X_train_major)

        gmm_scores_raw = gmm_major.score_samples(X_test)
        gmm_anomaly = -gmm_scores_raw

        calibrator_gmm = calibrate_scores(gmm_anomaly, y_test, method="sigmoid")
        prob_gmm = calibrator_gmm.predict_proba(gmm_anomaly.reshape(-1, 1))[:, 1] if hasattr(calibrator_gmm, "predict_proba") else calibrator_gmm.predict(gmm_anomaly.reshape(-1, 1))

        auc_gmm = roc_auc_score(y_test, prob_gmm)
        aupr_gmm = average_precision_score(y_test, prob_gmm)
        results_cc['majority']['GMM_Sigmoid'].append((auc_gmm, aupr_gmm))

        if fold == 1:
            print(f"  Fold 1 - OCSVM: AUC={auc_sigmoid:.4f}, GMM: AUC={auc_gmm:.4f}")

        # ---------------------------------
        # 2) Minority eğitimi: sadece fraud (y_train == 1)
        # ---------------------------------
        X_train_minor = X_train[y_train == 1]

        if len(X_train_minor) > 0:
            # -- One-Class SVM (Minority) --
            ocsvm_minor = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
            ocsvm_minor.fit(X_train_minor)

            svm_scores_minor = -ocsvm_minor.decision_function(X_test)
            calib_sigmoid_minor = calibrate_scores(svm_scores_minor, y_test, method="sigmoid")
            calib_isotonic_minor = calibrate_scores(svm_scores_minor, y_test, method="isotonic")

            prob_sigmoid_minor = calib_sigmoid_minor.predict_proba(svm_scores_minor.reshape(-1, 1))[:, 1] if hasattr(calib_sigmoid_minor, "predict_proba") else calib_sigmoid_minor.predict(svm_scores_minor.reshape(-1, 1))
            prob_isotonic_minor = calib_isotonic_minor.predict(svm_scores_minor.reshape(-1, 1))

            results_cc['minority']['OCSVM_Sigmoid'].append((roc_auc_score(y_test, prob_sigmoid_minor), average_precision_score(y_test, prob_sigmoid_minor)))
            results_cc['minority']['OCSVM_Isotonic'].append((roc_auc_score(y_test, prob_isotonic_minor), average_precision_score(y_test, prob_isotonic_minor)))

            # -- GMM (Minority) --
            gmm_minor = GaussianMixture(n_components=1, random_state=42)
            gmm_minor.fit(X_train_minor)

            gmm_scores_minor = -gmm_minor.score_samples(X_test)
            calib_gmm_minor = calibrate_scores(gmm_scores_minor, y_test, method="sigmoid")
            prob_gmm_minor = calib_gmm_minor.predict_proba(gmm_scores_minor.reshape(-1, 1))[:, 1] if hasattr(calib_gmm_minor, "predict_proba") else calib_gmm_minor.predict(gmm_scores_minor.reshape(-1, 1))

            results_cc['minority']['GMM_Sigmoid'].append((roc_auc_score(y_test, prob_gmm_minor), average_precision_score(y_test, prob_gmm_minor)))

    print(f"--- Credit Card Tekrar {r+1} tamamlandı ---")

print("\n✓ Tüm eğitimler tamamlandı!")


--- Credit Card: Tekrar 1/5 ---
  Fold 1 - OCSVM: AUC=0.9284, GMM: AUC=0.9480
--- Credit Card Tekrar 1 tamamlandı ---

--- Credit Card: Tekrar 2/5 ---
  Fold 1 - OCSVM: AUC=0.9623, GMM: AUC=0.9777
--- Credit Card Tekrar 2 tamamlandı ---

--- Credit Card: Tekrar 3/5 ---
  Fold 1 - OCSVM: AUC=0.9405, GMM: AUC=0.9521
--- Credit Card Tekrar 3 tamamlandı ---

--- Credit Card: Tekrar 4/5 ---
  Fold 1 - OCSVM: AUC=0.9395, GMM: AUC=0.9478
--- Credit Card Tekrar 4 tamamlandı ---

--- Credit Card: Tekrar 5/5 ---
  Fold 1 - OCSVM: AUC=0.9631, GMM: AUC=0.9678
--- Credit Card Tekrar 5 tamamlandı ---

✓ Tüm eğitimler tamamlandı!


## 6. Değerlendirme

In [9]:
def compute_mean_scores(score_list):
    """(AUC, AUPRC) tuple listesinden ortalama hesapla."""
    arr = np.array(score_list)
    return arr.mean(axis=0)

print("="*60)
print("CREDIT CARD - SONUÇ KARŞILAŞTIRMASI")
print("="*60)

for group in results_cc:
    print(f'\n>>> {group.upper()}')
    print(f"{'Model':20s} | {'AUC':10s} | {'AUPRC':10s}")
    print("-"*45)
    
    for method in results_cc[group]:
        scores = results_cc[group][method]
        if scores:
            mean_auc, mean_aupr = compute_mean_scores(scores)
            print(f'{method:20s} | {mean_auc:.4f}     | {mean_aupr:.4f}')
        else:
            print(f'{method:20s} | Sonuç yok')

CREDIT CARD - SONUÇ KARŞILAŞTIRMASI

>>> MAJORITY
Model                | AUC        | AUPRC     
---------------------------------------------
OCSVM                | Sonuç yok
OCSVM_Sigmoid        | 0.9449     | 0.3033
OCSVM_Isotonic       | 0.9547     | 0.3101
GMM                  | Sonuç yok
GMM_Sigmoid          | 0.9575     | 0.4949

>>> MINORITY
Model                | AUC        | AUPRC     
---------------------------------------------
OCSVM                | Sonuç yok
OCSVM_Sigmoid        | 0.4344     | 0.0041
OCSVM_Isotonic       | 0.6251     | 0.0048
GMM                  | Sonuç yok
GMM_Sigmoid          | 0.6614     | 0.0133


## 7. Sonuçları Kaydet

In [10]:
# Pickle olarak kaydet
os.makedirs("../pickled_storage", exist_ok=True)

with open("../pickled_storage/results_cc_gmm_ocsvm.pkl", "wb") as f:
    pickle.dump(results_cc, f)

print("✓ Sonuçlar kaydedildi: pickled_storage/results_cc_gmm_ocsvm.pkl")

✓ Sonuçlar kaydedildi: pickled_storage/results_cc_gmm_ocsvm.pkl
